# Brownian motion and Langevin equations: a hands-on comparison

This notebook builds two kinds of stochastic time series, fits models to them, and uses the fitted models to make **distributional forecasts**. The aim is not to guess one exact future path—an SDE cannot do that—but to learn the systematic part of the dynamics and the size of the remaining randomness.

We will work in this order:

1. Simulate and fit pure Brownian motion.
2. Simulate a constant-drift Langevin process.
3. Fit both a Brownian model and a Langevin model to that same Langevin data.
4. Simulate many futures from each fitted model and compare their prediction intervals with held-out observations.

## The two models

**Brownian motion** has no preferred direction:

$$dX_t = \sigma\,dW_t.$$

Its position wanders forever and its uncertainty grows with time. The parameter $\sigma$ controls how large the random steps are.

The simplest **overdamped Langevin equation** has a constant drift:

$$dX_t = v\,dt + \sigma\,dW_t.$$

Here $v$ is a constant average velocity: positive values create an upward tendency and negative values create a downward tendency. The random term still produces unpredictable departures from that average trend. Physically, this can represent a particle subject to a constant force in a highly damped environment.

In both models, $dW_t$ is not something we estimate point by point. It represents new, unpredictable Gaussian shocks. We infer the parameters that describe their scale and how the system reacts to them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)  # A fixed seed makes this lesson reproducible.
plt.style.use("seaborn-v0_8-whitegrid")

def simulate_brownian(x0, sigma, dt, n_steps, rng):
    """Simulate dX = sigma dW using Euler-Maruyama."""
    shocks = rng.normal(size=n_steps)
    increments = sigma * np.sqrt(dt) * shocks
    return np.r_[x0, x0 + np.cumsum(increments)]

def simulate_langevin(x0, drift, sigma, dt, n_steps, rng):
    """Simulate dX = drift dt + sigma dW with Euler-Maruyama."""
    shocks = rng.normal(size=n_steps)
    increments = drift * dt + sigma * np.sqrt(dt) * shocks
    return np.r_[x0, x0 + np.cumsum(increments)]


## Part 1 — Brownian motion

Euler–Maruyama turns the continuous-time Brownian equation into a simulation rule at a finite time step $\Delta t$:

$$X_{t+\Delta t}=X_t+\sigma\sqrt{\Delta t}Z_t, \qquad Z_t\sim\mathcal N(0,1).$$

The $Z_t$ values are independent random draws. A single run below is **one realization** of the process—not a universal path that Brownian motion must follow.

In [ ]:
dt = 0.02
n_steps = 2_000
time = np.arange(n_steps + 1) * dt

brownian_sigma_true = 0.8
brownian = simulate_brownian(x0=0.0, sigma=brownian_sigma_true, dt=dt,
                            n_steps=n_steps, rng=rng)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(time, brownian, lw=1.2, label="one Brownian realization")
ax.set(xlabel="time", ylabel="X(t)", title="A path can wander far even when its average increment is zero")
ax.legend();
plt.show()

### Fitting the Brownian noise scale from one path

For Brownian motion, increments satisfy $\Delta X_t\sim\mathcal N(0,\sigma^2\Delta t)$. Even though we have only one path, it contains many increments. The maximum-likelihood estimate is:

$$\hat\sigma=\sqrt{\frac{1}{n\Delta t}\sum_{i=1}^n(\Delta X_i)^2}. $$

This works because all increments share the same noise-scale parameter. We are *not* recovering future shocks; we are estimating the distribution from which shocks are drawn.

In [ ]:
brownian_increments = np.diff(brownian)
brownian_sigma_hat = np.sqrt(np.mean(brownian_increments**2) / dt)

print(f"True sigma:      {brownian_sigma_true:.3f}")
print(f"Estimated sigma: {brownian_sigma_hat:.3f}")

standardized_shocks = brownian_increments / (brownian_sigma_hat * np.sqrt(dt))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(standardized_shocks, bins=35, density=True, alpha=0.7, label="implied shocks")
z = np.linspace(-4, 4, 300)
ax.plot(z, np.exp(-z**2 / 2) / np.sqrt(2 * np.pi), color="black", label="standard normal density")
ax.set(title="Brownian-model residual check", xlabel="standardized increment")
ax.legend();
plt.show()

## Part 2 — Generate noisy dynamics with a constant tendency

Now we generate data from $dX_t=v\,dt+\sigma\,dW_t$. Every time step has the same average push, $v\Delta t$, plus a fresh random shock. Unlike Brownian motion, this process has a preferred direction on average. Unlike the OU process, it has no mean-reversion parameter to learn.

We will hold out the final part of this one realization. It acts as future data that the models do not get to see while fitting.

In [ ]:
drift_true = 0.18
langevin_sigma_true = 0.55
langevin = simulate_langevin(x0=-1.5, drift=drift_true, sigma=langevin_sigma_true,
                              dt=dt, n_steps=n_steps, rng=rng)

train_end = 1_500
train_time, test_time = time[:train_end + 1], time[train_end:]
train, test = langevin[:train_end + 1], langevin[train_end:]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train_time, train, label="training path")
ax.plot(test_time, test, color="tab:orange", label="held-out future")
ax.plot(time, -1.5 + drift_true * time, color="black", ls="--", lw=1, label="noise-free average trend")
ax.axvline(train_time[-1], color="grey", ls=":")
ax.set(xlabel="time", ylabel="X(t)", title="One constant-drift Langevin realization")
ax.legend(ncol=2);
plt.show()

## Part 3 — Fit a Brownian model and a Langevin model to the same training data

The Brownian fit can only say: “future changes are random with this estimated scale.” It cannot represent mean reversion.

For this Langevin fit, Euler–Maruyama gives a particularly simple update:

$$\Delta X_t = v\Delta t + \sigma\sqrt{\Delta t}Z_t. $$

The average increment estimates the deterministic part: $\hat v=\operatorname{mean}(\Delta X)/\Delta t$. After subtracting that average increment, the remaining spread estimates $\sigma$. This is much simpler than the OU fit because there is no state-dependent drift function to infer.

In [ ]:
# Brownian model: no drift, so all changes are treated as noise.
train_increments = np.diff(train)
sigma_bm_fit = np.sqrt(np.mean(train_increments**2) / dt)

# Constant-drift Langevin model: the mean increment is the deterministic drift.
drift_hat = np.mean(train_increments) / dt
langevin_residuals = train_increments - drift_hat * dt
sigma_langevin_hat = np.sqrt(np.mean(langevin_residuals**2) / dt)

print("True data-generating parameters")
print(f"  drift = {drift_true:.3f}, sigma = {langevin_sigma_true:.3f}")
print("\nFitted Brownian model")
print(f"  drift = 0.000 (assumed), sigma = {sigma_bm_fit:.3f}")
print("\nFitted Langevin model")
print(f"  drift = {drift_hat:.3f}, sigma = {sigma_langevin_hat:.3f}")

A good fit does not require the estimates to equal the true values exactly: we generated only one finite random path. The question is whether the fitted parameters describe its patterns well enough to make useful probabilistic forecasts.

A useful diagnostic is to compare the inferred standardized shocks with a standard normal distribution. If the model is appropriate, they should be roughly centered at zero, have spread near one, and not show obvious structure over time.

In [ ]:
langevin_z = langevin_residuals / (sigma_langevin_hat * np.sqrt(dt))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(langevin_z, bins=30, density=True, alpha=0.7)
axes[0].plot(z, np.exp(-z**2 / 2) / np.sqrt(2 * np.pi), color="black")
axes[0].set(title="Distribution of inferred shocks", xlabel="standardized residual")
axes[1].plot(train_time[1:], langevin_z, lw=0.7)
axes[1].axhline(0, color="black", lw=1)
axes[1].set(title="Inferred shocks over time", xlabel="time", ylabel="standardized residual")
plt.tight_layout()
plt.show()

## Part 4 — Forecast with many simulated paths

Neither model returns *the* exact held-out path. Instead, each produces many possible futures from the last training value. At each future time, we summarize those simulations with:

- the median (50th percentile), our central forecast;
- the 5th and 95th percentiles, a 90% prediction interval.

The Brownian model's interval keeps spreading and its central forecast stays at the initial value. The constant-drift Langevin model's median instead follows its fitted straight-line average trend, while retaining random variation.

In [ ]:
forecast_steps = len(test) - 1
n_paths = 2_000
x_last = train[-1]

bm_paths = np.vstack([
    simulate_brownian(x_last, sigma_bm_fit, dt, forecast_steps, rng)
    for _ in range(n_paths)
])
langevin_paths = np.vstack([
    simulate_langevin(x_last, drift_hat, sigma_langevin_hat, dt, forecast_steps, rng)
    for _ in range(n_paths)
])

def forecast_bands(paths):
    return np.quantile(paths, [0.05, 0.50, 0.95], axis=0)

bm_low, bm_median, bm_high = forecast_bands(bm_paths)
langevin_low, langevin_median, langevin_high = forecast_bands(langevin_paths)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, low, median, high, title in [
    (axes[0], bm_low, bm_median, bm_high, "Brownian model fit to drifted data"),
    (axes[1], langevin_low, langevin_median, langevin_high, "Constant-drift Langevin model"),
]:
    ax.plot(test_time, test, color="black", lw=1.5, label="actual held-out path")
    ax.plot(test_time, median, color="tab:blue", label="median forecast")
    ax.fill_between(test_time, low, high, color="tab:blue", alpha=0.22, label="90% prediction interval")
    ax.set(title=title, xlabel="future time")
axes[0].set_ylabel("X(t)")
axes[1].legend(loc="best")
plt.tight_layout()
plt.show()

bm_coverage = np.mean((test >= bm_low) & (test <= bm_high))
langevin_coverage = np.mean((test >= langevin_low) & (test <= langevin_high))
print(f"Fraction of held-out points inside Brownian 90% band: {bm_coverage:.1%}")
print(f"Fraction of held-out points inside Langevin 90% band: {langevin_coverage:.1%}")

## What this exercise establishes

- A single observed path still contains many time increments, which can inform shared parameters such as drift and diffusion.
- A Brownian model captures random movement but assumes its average increment is zero.
- A Langevin model separates a systematic tendency (the drift) from irreducible random shocks (the diffusion).
- Forecasts from an SDE are distributions of possible futures. The interval is a **prediction interval**, not a promise that the actual path will be inside it.
- This worked because we deliberately made data that match the constant-drift assumptions: continuous state, small time steps, Gaussian independent shocks, and stable parameters. Real data can violate any of these—for example through changing trends, jumps, autocorrelated noise, or measurement error.

### Suggested experiments

1. Change `drift_true` from positive to negative. How does that change the median forecast?
2. Change `langevin_sigma_true`. Which plots respond to larger process noise?
3. Make `train_end` much smaller. Which fitted parameters become unstable?
4. Add occasional large jumps to the generated path. Do the inferred shocks still look Gaussian?
5. Next step: add mean reversion, $-\theta(X-\mu)$, to make an OU process. Only after that: replace the hand-specified drift with a neural-network drift and ask whether the extra flexibility improves held-out predictive performance.